# Homework Option A: Build a Toy 2D Diffusion Model

Time: up to 60 minutes

This notebook builds a complete diffusion model on 2D points instead of images.
Same forward process from Part 1, same reverse process from Part 2, just a tiny MLP instead of a U-Net. Runs on CPU in under a minute.
Look for `# TODO` comments to complete it.
See `Homework_OptionA_ToyDiffusion.md` for full instructions.

In [ ]:
!pip install -q scikit-learn

In [ ]:
import torch
import torch.nn as nn
from torch.optim import Adam
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons

torch.manual_seed(0)
np.random.seed(0)
print('Ready.')

---

## Step 1: The Dataset

We use `make_moons`: two crescent-shaped clusters of 2D points. This is already working, just run it.

In [ ]:
X, labels = make_moons(n_samples=2000, noise=0.06)
X = (X - X.mean(0)) / X.std(0)   # normalize to roughly unit variance
data = torch.tensor(X, dtype=torch.float32)

plt.figure(figsize=(4, 4))
plt.scatter(X[:, 0], X[:, 1], s=5, alpha=0.5)
plt.title('Toy dataset: two moons')
plt.axis('equal')
plt.show()

print(f'{len(data)} points loaded.')

---

## Step 2: Forward Diffusion (recap from Part 1)

Same noise schedule and closed-form `q` function as Part 1, just for 2D points instead of images. Already working.

In [ ]:
T = 200
beta  = torch.linspace(1e-4, 0.02, T)
alpha = 1.0 - beta
alpha_bar = torch.cumprod(alpha, dim=0)
sqrt_alpha_bar           = torch.sqrt(alpha_bar)
sqrt_one_minus_alpha_bar = torch.sqrt(1 - alpha_bar)
sqrt_alpha_inv           = torch.sqrt(1 / alpha)
pred_noise_coeff         = (1 - alpha) / sqrt_one_minus_alpha_bar

def q(x_0, t):
    noise = torch.randn_like(x_0)
    x_t = sqrt_alpha_bar[t][:, None] * x_0 + sqrt_one_minus_alpha_bar[t][:, None] * noise
    return x_t, noise

# Visualize forward diffusion at a few timesteps
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
t_values = [0, 50, 100, 150, 199]
x0_sample = data[:500]
for ax, t_val in zip(axes, t_values):
    t_batch = torch.full((len(x0_sample),), t_val, dtype=torch.long)
    x_t, _ = q(x0_sample, t_batch)
    ax.scatter(x_t[:, 0], x_t[:, 1], s=3, alpha=0.4)
    ax.set_title(f't={t_val}')
    ax.axis('equal')
    ax.axis('off')
plt.suptitle('Forward diffusion: points dissolving into noise')
plt.tight_layout()
plt.show()

---

## Step 3: The Model

A tiny MLP plays the role of the U-Net here. It takes a noisy point and a timestep, and predicts the noise that was added. Already working.

In [ ]:
class TinyMLP(nn.Module):
    def __init__(self, hidden_dim=128, t_embed_dim=16):
        super().__init__()
        self.time_embed = nn.Sequential(
            nn.Linear(1, t_embed_dim), nn.GELU(),
            nn.Linear(t_embed_dim, t_embed_dim),
        )
        self.net = nn.Sequential(
            nn.Linear(2 + t_embed_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, 2),
        )

    def forward(self, x, t_norm):
        t_emb = self.time_embed(t_norm.view(-1, 1))
        h = torch.cat([x, t_emb], dim=1)
        return self.net(h)

model = TinyMLP()
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

---

## Step 4: Training

8000 training steps on 2000 tiny 2D points takes about 15-20 seconds on CPU. Each step uses a fresh random batch, so this is closer to 8000 mini batches than 8000 full passes over the data. Already working.

In [ ]:
optimizer = Adam(model.parameters(), lr=1e-3)
STEPS = 8000
BATCH_SIZE = 256

model.train()
for step in range(STEPS):
    idx = torch.randint(0, len(data), (BATCH_SIZE,))
    x_0 = data[idx]
    t   = torch.randint(0, T, (BATCH_SIZE,))
    x_noisy, noise = q(x_0, t)
    pred = model(x_noisy, t.float() / T)
    loss = nn.functional.mse_loss(pred, noise)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (step + 1) % 2000 == 0:
        print(f'Step {step+1}/{STEPS} | loss: {loss.item():.4f}')

model.eval()
print('Training done.')

---

## Step 5: Reverse Diffusion (your turn)

The forward process and the model are already working. Now implement the reverse diffusion step: the same formula from Part 2, just for 2D points.

Right now this function is a placeholder that does nothing useful (it just returns the input unchanged). Fix the `# TODO` line.

In [ ]:
def reverse_step(x_t, t, e_t):
    """
    One step of reverse diffusion. Same formula as Part 2's reverse_q.
    x_t : noisy points at timestep t, shape [B, 2]
    t   : current timestep per sample, shape [B] (all the same value)
    e_t : predicted noise from the model, shape [B, 2]
    Returns the points one step less noisy.
    """
    sqrt_alpha_inv_t = sqrt_alpha_inv[t][:, None]
    coeff_t          = pred_noise_coeff[t][:, None]

    # TODO: compute u_t, the denoised estimate (same formula as Part 2's reverse_q)
    u_t = x_t   # placeholder, replace this line

    if t[0] == 0:
        return u_t
    beta_prev = beta[t - 1][:, None]
    return u_t + torch.sqrt(beta_prev) * torch.randn_like(x_t)

print('reverse_step defined. Currently a placeholder, fix the TODO above.')

<details>
<summary><b>Click to show the solution</b></summary>

```python
u_t = sqrt_alpha_inv_t * (x_t - coeff_t * e_t)
```
</details>

---

## Step 6: Generate and Visualize

This samples from pure noise using your `reverse_step`, and shows the points forming back into the moon shapes. Already working, once Step 5 is fixed.

In [ ]:
@torch.no_grad()
def sample(model, n=500):
    x = torch.randn(n, 2)
    snapshots = {}
    for i in reversed(range(T)):
        t_batch = torch.full((n,), i, dtype=torch.long)
        e_t = model(x, t_batch.float() / T)
        x = reverse_step(x, t_batch, e_t)
        if i % (T // 5) == 0:
            snapshots[i] = x.clone()
    snapshots[0] = x.clone()
    return x, snapshots

final, snapshots = sample(model, n=500)

fig, axes = plt.subplots(1, len(snapshots), figsize=(15, 3))
for ax, (t_val, pts) in zip(axes, sorted(snapshots.items(), reverse=True)):
    ax.scatter(pts[:, 0], pts[:, 1], s=3, alpha=0.4, color='steelblue')
    ax.set_title(f't={t_val}')
    ax.axis('equal')
    ax.axis('off')
plt.suptitle('Reverse diffusion: noise forming back into moons')
plt.tight_layout()
plt.show()

---

## Step 7: Try It Yourself

Swap the dataset for a different 2D shape and retrain. A couple of options from `sklearn.datasets` (uncomment one and rerun from Step 1):

```python
# from sklearn.datasets import make_circles
# X, labels = make_circles(n_samples=2000, noise=0.05, factor=0.5)

# from sklearn.datasets import make_blobs
# X, labels = make_blobs(n_samples=2000, centers=3, cluster_std=0.3)
```

Or write your own point generator. Anything that returns an `[N, 2]` array works.

### Bonus (optional, if you have extra time)

`make_moons` already gives you 2 class labels. Try adding simple class conditioning: extend `TinyMLP` to also take a one-hot class vector as input (concatenate it alongside the time embedding), train using the labels you already have, then generate each moon separately. This is the same idea as Part 3's classifier-free guidance, just on 2D points instead of flowers.

---

## Your Report

Answer these questions in this cell:

**1. Which dataset shape did you try in Step 7, and how did the result compare to the two moons?**

_(your answer here)_

**2. Look at the Step 6 visualization. At which timestep does the shape become recognizable? Does this match what you would expect from the noise schedule?**

_(your answer here)_

**3. If you did the bonus: did conditioning on class actually separate the two moons during generation? If you did not do the bonus: what do you think would happen if you ran generation with no conditioning at all on a 3-cluster dataset?**

_(your answer here)_